In [1]:
import torch, torch.optim as optim, numpy as np, gc, warnings
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from IPython.utils import io
from reader import prepare_qsm_dataset
from loader import get_slice_level_data
from util import seed_everything, mask_crop as mask_crop_fn, mean_std_str, compute_comprehensive_metrics, compare_auc_significance
from train import calibrate_balanced
from validate import ClinicalTransformer_cv, ResidualSpectralViT_cv
from networks import FocalLoss, PassThrough, ClinicalTransformer, SpectralViT, SpatialViT, ResidualSpectral, ResidualSpatial

warnings.filterwarnings("ignore", category=UserWarning, message="Default upsampling behavior")

# Configuration
device = torch.device("cuda:4" if torch.cuda.is_available() else "cpu")
N_FOLDS, EPOCHS, JITTER_STD, IMG_AUG_STD, TARGET_DIM = 5, 100, 0.1, 0.05, 128
seed_everything(0)

# Load datasets
dataset_train = prepare_qsm_dataset('MSW', '/media/mts_dbs/dbs/all/nii/qsm_115/im', '/media/mts_dbs/dbs/all/nii/seg_ps/', '/data/Ali/RadDBS-QSM/data/docs/dbs_03292024.csv', './msw_cache_6d_cv.pt', load_cache=True, mask_crop_fn=mask_crop_fn, cv_pad=False)
dataset_test = prepare_qsm_dataset('CHH', '/media/mts_dbs/chh/nii/qsm/', '/media/mts_dbs/chh/roi/', '/media/mts_dbs/chh/xlsx/chh_subjects_table1_20240729.csv', './chh_cache_6d_cv.pt', load_cache=True, mask_crop_fn=mask_crop_fn, cv_pad=False)

# Extract Arrays
X_full_slices, X_full_clin, y_full_slices, full_subj_map = get_slice_level_data(dataset_train, TARGET_DIM, include_unlabeled=True)
labeled_mask = (y_full_slices != -1)
X_tr_slices, X_tr_clin_raw, y_tr_slices, tr_subj_map = X_full_slices[labeled_mask], X_full_clin[labeled_mask], y_full_slices[labeled_mask], full_subj_map[labeled_mask]
X_te_slices, X_te_clin_raw, y_te_slices, te_subj_map = get_slice_level_data(dataset_test, TARGET_DIM)

# Harmonization & Weights
scaler_train = StandardScaler()
X_tr_clin = scaler_train.fit_transform(X_tr_clin_raw)
scaler_test = StandardScaler()
X_te_clin = scaler_test.fit(X_te_clin_raw[((X_te_clin_raw.shape[0])//4):,:]).transform(X_te_clin_raw)
NEG_WEIGHT = float(sum(y_tr_slices == 1) // sum(y_tr_slices == 0))
GAMMA = NEG_WEIGHT

# Hyperparameter selection
unique_subjs = np.unique(tr_subj_map)
y_unique = np.array([y_tr_slices[tr_subj_map == s][0] for s in unique_subjs])
outer_skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
criterion = FocalLoss(gamma=GAMMA)

SELECTED_POS_WEIGHT = ClinicalTransformer_cv(model_class=ClinicalTransformer, model_kwargs={'n_inputs': X_tr_clin.shape[1]}, pos_weight_grid=[0.1, 0.25, 0.5, 0.75, 1.0], outer_skf=outer_skf, unique_subjs=unique_subjs, y_unique=y_unique, tr_subj_map=tr_subj_map, X_tr_clin=X_tr_clin, y_tr_slices=y_tr_slices, NEG_WEIGHT=NEG_WEIGHT, criterion=criterion, JITTER_STD=JITTER_STD, EPOCHS=EPOCHS, device=device)

model_classes = {'clinical': ClinicalTransformer, 'vit': SpectralViT, 'wrapper': ResidualSpectral}
model_kwargs = {'vit': {'n_heads': 1, 'n_layers': 1, 'embed_dim': 32, 'use_input_proj': False, 'use_pos_embed': False, 'use_layer_norm': False, 'pooling': 'flatten'}}

N_PCA_COMPONENTS = ResidualSpectralViT_cv(model_classes=model_classes, model_kwargs=model_kwargs, pca_components_grid=[16, 32, 64, 128], outer_skf=outer_skf, unique_subjs=unique_subjs, y_unique=y_unique, tr_subj_map=tr_subj_map, X_tr_slices=X_tr_slices, X_tr_clin=X_tr_clin, y_tr_slices=y_tr_slices, NEG_WEIGHT=NEG_WEIGHT, SELECTED_POS_WEIGHT=SELECTED_POS_WEIGHT, criterion=criterion, JITTER_STD=JITTER_STD, EPOCHS=EPOCHS, device=device)

# Cross-validation
print("Training...")
clinical_fold_weights = []
eval_skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
all_cv_probs_ct, all_cv_probs_sp, all_cv_probs_va, all_cv_labels = [], [], [], []
fold_preds_ct, fold_preds_sp, fold_preds_spatial = [], [], []
fold_thresholds_ct, fold_thresholds_sp, fold_thresholds_va = [], [], []

for fold_idx, (train_subj_idx, val_subj_idx) in enumerate(eval_skf.split(unique_subjs, y_unique)):
    print(f"Fold {fold_idx + 1}/{N_FOLDS}")
    train_mask, val_mask = np.isin(tr_subj_map, unique_subjs[train_subj_idx]), np.isin(tr_subj_map, unique_subjs[val_subj_idx])
    
    f_img_scaler = StandardScaler().fit(X_tr_slices[train_mask])
    f_pca = PCA(n_components=N_PCA_COMPONENTS, random_state=0, whiten=True).fit(f_img_scaler.transform(X_tr_slices[train_mask]))
    
    X_train_pca, X_train_clin, X_train_img, y_train = f_pca.transform(f_img_scaler.transform(X_tr_slices[train_mask])), X_tr_clin[train_mask], X_tr_slices[train_mask], y_tr_slices[train_mask]
    X_val_pca, X_val_clin, X_val_img, y_val = f_pca.transform(f_img_scaler.transform(X_tr_slices[val_mask])), X_tr_clin[val_mask], X_tr_slices[val_mask], y_tr_slices[val_mask]
    X_test_pca, X_test_clin, X_test_img = f_pca.transform(f_img_scaler.transform(X_te_slices)), X_te_clin, X_te_slices
    
    X_train_clin_t, X_train_pca_t = torch.tensor(X_train_clin, dtype=torch.float32).to(device), torch.tensor(X_train_pca, dtype=torch.float32).to(device)
    X_train_img_t, y_train_t = torch.tensor(X_train_img).view(-1, 1, 128, 128).float().to(device), torch.tensor(y_train, dtype=torch.float32).to(device)
    w = torch.where(y_train_t == 0, torch.tensor(NEG_WEIGHT, device=device), torch.tensor(SELECTED_POS_WEIGHT, device=device))

    m_ct = ClinicalTransformer(n_inputs=X_tr_clin.shape[1]).to(device)
    opt_ct = optim.Adam(m_ct.parameters(), lr=1e-4)
    for _ in range(EPOCHS):
        m_ct.train(); opt_ct.zero_grad()
        loss = criterion(m_ct(X_train_clin_t + torch.randn_like(X_train_clin_t) * JITTER_STD), y_train_t, weight=w)
        loss.backward(); opt_ct.step()
    
    m_ct.eval(); [p.requires_grad_(False) for p in m_ct.parameters()]
    clinical_fold_weights.append(m_ct.state_dict())

    m_sp = ResidualSpectral(m_ct, SpectralViT(n_inputs=N_PCA_COMPONENTS, n_heads=1, n_layers=1, embed_dim=32, use_input_proj=False, use_pos_embed=False, use_layer_norm=False, pooling='flatten')).to(device)
    m_va = ResidualSpatial(m_ct, SpatialViT(size=128, patch_size=16, embed_dim=32, n_heads=1, n_layers=1, dropout=0.1, is_2d=True, use_cls_token=False, use_layer_norm=False)).to(device)
    
    opt_va = optim.Adam(m_va.m_res.parameters(), lr=1e-4)
    for _ in range(EPOCHS):
        m_va.train(); opt_va.zero_grad()
        loss = criterion(m_va(X_train_img_t + torch.randn_like(X_train_img_t) * IMG_AUG_STD, X_train_clin_t), y_train_t, weight=w)
        loss.backward(); opt_va.step()

    with torch.no_grad():
        v_probs_ct, v_probs_sp, v_probs_va, v_labels = [], [], [], []
        for s in unique_subjs[val_subj_idx]:
            m = (tr_subj_map[val_mask] == s); v_labels.append(y_val[m][0])
            v_probs_ct.append(m_ct(torch.tensor(X_val_clin[m], dtype=torch.float32).to(device)).mean().item())
            v_probs_sp.append(m_sp(torch.tensor(X_val_pca[m], dtype=torch.float32).to(device), torch.tensor(X_val_clin[m], dtype=torch.float32).to(device)).mean().item())
            v_probs_va.append(m_va(torch.tensor(X_val_img[m]).view(-1, 1, 128, 128).float().to(device), torch.tensor(X_val_clin[m], dtype=torch.float32).to(device)).mean().item())

        fold_thresholds_ct.append(calibrate_balanced(PassThrough(), None, np.array(v_probs_ct), np.array(v_labels), 'cpu'))
        fold_thresholds_sp.append(calibrate_balanced(PassThrough(), None, np.array(v_probs_sp), np.array(v_labels), 'cpu'))
        fold_thresholds_va.append(calibrate_balanced(PassThrough(), None, np.array(v_probs_va), np.array(v_labels), 'cpu'))
        all_cv_probs_ct.extend(v_probs_ct); all_cv_probs_sp.extend(v_probs_sp); all_cv_probs_va.extend(v_probs_va); all_cv_labels.extend(v_labels)

        t_probs_ct, t_probs_sp, t_probs_va = [], [], []
        for s in np.unique(te_subj_map):
            m = (te_subj_map == s)
            t_probs_ct.append(m_ct(torch.tensor(X_test_clin[m], dtype=torch.float32).to(device)).mean().item())
            t_probs_sp.append(m_sp(torch.tensor(X_test_pca[m], dtype=torch.float32).to(device), torch.tensor(X_test_clin[m], dtype=torch.float32).to(device)).mean().item())
            t_probs_va.append(m_va(torch.tensor(X_test_img[m]).view(-1, 1, 128, 128).float().to(device), torch.tensor(X_test_clin[m], dtype=torch.float32).to(device)).mean().item())
        fold_preds_ct.append(t_probs_ct); fold_preds_sp.append(t_probs_sp); fold_preds_spatial.append(t_probs_va)

th_ct, th_sp, th_va = np.mean(fold_thresholds_ct), np.mean(fold_thresholds_sp), np.mean(fold_thresholds_va)
y_test_labels = np.array([y_te_slices[te_subj_map == s][0] for s in np.unique(te_subj_map)])
cv_probs = {"Clinical": all_cv_probs_ct, "Spectral": all_cv_probs_sp, "Spatial": all_cv_probs_va}
test_probs = {"Clinical": np.mean(fold_preds_ct, axis=0), "Spectral": np.mean(fold_preds_sp, axis=0), "Spatial": np.mean(fold_preds_spatial, axis=0)}
thresholds = {"Clinical": th_ct, "Spectral": th_sp, "Spatial": th_va}


Preparing MSW dataset (Forced 6-dim alignment) 
Validating MSW CSV mapping...
--- MSW CSV RAW MEANS ---
  > Age     : 62.13
  > Sex     : 0.26
  > Dur     : 8.47
  > LEDD    : 989.80
  > Off-Pre : 45.91
  > On-Pre  : 19.60

Final MSW Breakdown:
 - Unique Subjects on Disk: 111
 - Labeled Responders (1): 61
 - Labeled Non-Responders (0): 5
 - Unlabeled subjects (-1): 45
 - Verified Realized Means (Matched Data Only):
    > Age     : 63.08
    > Sex     : 0.26
    > Dur     : 8.44
    > LEDD    : 1003.95
    > Off-Pre : 45.62
    > On-Pre  : 19.55

❌ FULL MISSING LIST (45 subjects):
  [3, 4, 5, 8, 12, 13, 14, 17, 18, 21, 22, 24, 25, 27, 28, 31, 32, 34, 35, 37, 39, 40, 41, 42, 49, 50, 52, 54, 57, 61, 65, 67, 74, 76, 81, 82, 84, 88, 89, 94, 99, 101, 104, 105, 116]
Loaded cache with 7790 slices.

Preparing CHH dataset (Forced 6-dim alignment) 
Validating CHH CSV mapping...
--- CHH CSV RAW MEANS ---
  > Age     : 63.13
  > Sex     : 0.46
  > Dur     : 8.54
  > LEDD    : 728.27
  > Off-Pre : 

In [2]:
from torch.cuda.amp import GradScaler, autocast
from networks import AttentionUNet

# Flush and initialize
all_cv_probs_un, fold_preds_un, fold_ths_un, scaler_amp = [], [], [], GradScaler()

for fold, (t_subj_idx, v_subj_idx) in enumerate(eval_skf.split(unique_subjs, y_unique)):
    print(f"Fold {fold+1} Attention U-Net: ", end='', flush=True)
    
    # 1. Setup Data & Clinical Model (Unchanged)
    t_mask = np.isin(tr_subj_map, unique_subjs[t_subj_idx])
    scaler_c = StandardScaler().fit(X_tr_clin[t_mask])
    m_ct_fold = ClinicalTransformer(n_inputs=X_tr_clin.shape[1]).to(device)
    m_ct_fold.load_state_dict(clinical_fold_weights[fold]); m_ct_fold.eval()
    
    with torch.no_grad():
        xt_c_gpu = torch.tensor(scaler_c.transform(X_tr_clin[t_mask]), dtype=torch.float32).to(device)
        t_clin_logits = m_ct_fold(xt_c_gpu, return_logit=True).detach()

    xt_i_gpu = torch.tensor(X_tr_slices[t_mask]).view(-1, 1, 128, 128).float().to(device)
    yt_gpu = torch.tensor(y_tr_slices[t_mask], dtype=torch.float32).to(device)
    pos_idx, neg_idx = torch.where(yt_gpu == 1)[0], torch.where(yt_gpu == 0)[0]
    
    batch_size, half_batch = 64, 32
    num_batches = len(pos_idx) // half_batch

    # 2. Replicate FastUNet Call Exactly
    m_un_res = AttentionUNet(
        in_channels=1, 
        base_channels=16, # Creates 16->32->64 pipeline
        fast_mode=True    # Replicates pooling and SpatialAttention (1+att) logic
    ).to(device)
    optimizer = optim.Adam(m_un_res.parameters(), lr=5e-4)
   
    # 3. Training Loop
    for epoch in range(EPOCHS):
        m_un_res.train()
        shuffled_neg = neg_idx[torch.randperm(len(neg_idx))]
        
        for i in range(num_batches):
            n_ids = shuffled_neg[i*half_batch : (i+1)*half_batch]
            p_ids = pos_idx[torch.randint(0, len(pos_idx), (half_batch,))]
            b_idx = torch.cat([p_ids, n_ids])
            
            optimizer.zero_grad(set_to_none=True) 
            with autocast():
                # Correct logit summation
                res_logit = m_un_res(xt_i_gpu[b_idx] + torch.randn_like(xt_i_gpu[b_idx]) * IMG_AUG_STD)
                joint_logit = res_logit + t_clin_logits[b_idx]
                
                bce_loss = F.binary_cross_entropy_with_logits(joint_logit, yt_gpu[b_idx], reduction='none')
                w_batch = torch.where(yt_gpu[b_idx] == 0, torch.tensor(NEG_WEIGHT, device=device), torch.tensor(SELECTED_POS_WEIGHT, device=device))
                loss = (w_batch * (1 - torch.exp(-bce_loss))**GAMMA * bce_loss).mean()
            
            scaler_amp.scale(loss).backward()
            scaler_amp.step(optimizer)
            scaler_amp.update()
            
        if (epoch + 1) % 25 == 0: print(f'{epoch+1}..', end='', flush=True)
    print('Done.')

    # 4. Inference (Unchanged)
    m_un_res.eval()
    with torch.no_grad():
        v_un, v_lbls = [], []
        for s_id in unique_subjs[v_subj_idx]:
            sm = (tr_subj_map == s_id)
            c_v = torch.tensor(scaler_c.transform(X_tr_clin[sm]), dtype=torch.float32).to(device)
            i_v = torch.tensor(X_tr_slices[sm]).view(-1, 1, 128, 128).float().to(device)
            l_c = m_ct_fold(c_v, return_logit=True)
            v_un.append(torch.sigmoid(l_c + m_un_res(i_v)).mean().item())
            v_lbls.append(y_tr_slices[sm][0])
            
        fold_ths_un.append(calibrate_balanced(PassThrough(), None, np.array(v_un), np.array(v_lbls), 'cpu'))
        all_cv_probs_un.extend(v_un)

        t_un = []
        for s_id in np.unique(te_subj_map):
            sm = (te_subj_map == s_id)
            c_t = torch.tensor(scaler_c.transform(X_te_clin[sm]), dtype=torch.float32).to(device)
            i_t = torch.tensor(X_te_slices[sm]).view(-1, 1, 128, 128).float().to(device)
            l_c_t = m_ct_fold(c_t, return_logit=True)
            t_un.append(torch.sigmoid(l_c_t + m_un_res(i_t)).mean().item())
        fold_preds_un.append(t_un)
    torch.cuda.empty_cache()

# Final Results
cv_probs["Attention U-Net"] = all_cv_probs_un
test_probs["Attention U-Net"] = np.mean(fold_preds_un, axis=0)
thresholds["Attention U-Net"] = np.mean(fold_ths_un)

Fold 1 Attention U-Net: 25..50..75..100..Done.
Fold 2 Attention U-Net: 25..50..75..100..Done.
Fold 3 Attention U-Net: 25..50..75..100..Done.
Fold 4 Attention U-Net: 25..50..75..100..Done.
Fold 5 Attention U-Net: 25..50..75..100..Done.


In [3]:
# Fold-wise predictions
external_fold_preds_all = {
    "Clinical": fold_preds_ct,
    "PCA+LR": [], #fold_preds_lr,
    "PCA+MLP": [], #fold_preds_mlp,
    "Spectral": fold_preds_sp,
    "Spatial": fold_preds_spatial,
    "Attention U-Net": fold_preds_un
}


def get_cv_fold_metrics(name):
    probs = np.array(cv_probs[name])
    labels = np.array(all_cv_labels)
    fold_indices = np.array_split(np.arange(len(labels)), 5)
    
    fold_results = []
    for idxs in fold_indices:
        m = compute_comprehensive_metrics(labels[idxs], probs[idxs], thresholds[name])
        fold_results.append(m)
    return fold_results

def print_table(title, labels, prob_dict, thresh_dict, fold_data_dict, is_cv=False):
    print(f"\n{title}")
    header = f"{'Model':<16} | {'AUC':<14} | {'B-Acc':<14} | {'Spec':<14} | {'F1':<14}"
    print("-" * len(header))
    print(header)
    print("-" * len(header))

    models = ["Clinical", "PCA+LR", "PCA+MLP", "Spectral", "Spatial", "Attention U-Net"]

    for name in models:
        if name not in prob_dict: continue
        m_mean = compute_comprehensive_metrics(np.array(labels), np.array(prob_dict[name]), thresh_dict[name])
        if is_cv:
            f_metrics = get_cv_fold_metrics(name)
        else:
            f_metrics = []
            for f_idx in range(len(fold_data_dict[name])):
                f_metrics.append(compute_comprehensive_metrics(
                    np.array(labels), 
                    np.array(fold_data_dict[name][f_idx]), 
                    thresh_dict[name]
                ))
        
        stds = {k: np.std([f[k] for f in f_metrics]) for k in m_mean.keys()}
        def fmt(key):
            return f"{m_mean[key]:.3f}±{stds[key]:.3f}"
        print(f"{name:<16} | {fmt('AUC'):<14} | {fmt('B-Acc'):<14} | {fmt('Spec'):<14} | {fmt('F1'):<14}")

    # Significance test
    print("\nStatistical Significance (Incremental value over Clinical):")
    for name in models[1:]:
        if name in prob_dict:
            diff, p = compare_auc_significance(np.array(labels), np.array(prob_dict["Clinical"]), np.array(prob_dict[name]))
            sig = "*" if p < 0.05 else "n.s."
            print(f"  {name:<16}: ΔAUC {diff:+.3f}, p={p:.4f} ({sig})")

# Internal validation
print_table(
    "INTERNAL VALIDATION (5-Fold CV: Mean ± Inter-fold SD)", 
    all_cv_labels, cv_probs, thresholds, None, is_cv=True
)

# External test
print_table(
    "EXTERNAL TEST PERFORMANCE (CHH Dataset: Mean ± Inter-fold SD)", 
    y_test_labels, test_probs, thresholds, external_fold_preds_all, is_cv=False
)


INTERNAL VALIDATION (5-Fold CV: Mean ± Inter-fold SD)
------------------------------------------------------------------------------------
Model            | AUC            | B-Acc          | Spec           | F1            
------------------------------------------------------------------------------------
Clinical         | 0.821±0.080    | 0.745±0.070    | 0.794±0.115    | 0.812±0.025   
Spectral         | 0.810±0.063    | 0.734±0.043    | 0.797±0.105    | 0.795±0.061   
Spatial          | 0.824±0.065    | 0.749±0.052    | 0.786±0.080    | 0.824±0.030   
Attention U-Net  | 0.872±0.036    | 0.773±0.057    | 0.814±0.063    | 0.837±0.068   

Statistical Significance (Incremental value over Clinical):
  Spectral        : ΔAUC -0.010, p=1.0000 (n.s.)
  Spatial         : ΔAUC +0.004, p=0.0000 (*)
  Attention U-Net : ΔAUC +0.051, p=0.0000 (*)

EXTERNAL TEST PERFORMANCE (CHH Dataset: Mean ± Inter-fold SD)
------------------------------------------------------------------------------------
